In [1]:
import os
import cdsapi

# Local data folder for ERA5 outputs
DATA_DIR = '../../data/ERA5'
os.makedirs(DATA_DIR, exist_ok=True)

c = cdsapi.Client()

c.retrieve(
        'reanalysis-era5-single-levels-monthly-means',
        {
            'product_type': 'monthly_averaged_reanalysis',
            'variable': [
                'sea_surface_temperature',
                'mean_sea_level_pressure'  # OLR
            ],
            'year': str(1980),
            'month': [f'{m:02d}' for m in range(1, 13)],
            'time': '00:00',
            'grid': [2.0, 2.0],
            'area': [30, 120, -30, -80],  # N, W, S, E
            'format': 'netcdf'
        },
        os.path.join(DATA_DIR, f'era5_singlelevels_{1980}.nc')
    )


2026-06-25 14:08:22,265 INFO Request ID is 9c0dc560-c58f-43e5-b4d0-527128915bfd
2026-06-25 14:08:22,502 INFO status has been updated to accepted
2026-06-25 14:08:46,160 INFO status has been updated to running
2026-06-25 14:08:57,803 INFO status has been updated to successful


ae805bf4b3e6291569a51c20f55eebfc.nc:   0%|          | 0.00/136k [00:00<?, ?B/s]

'../../data/ERA5\\era5_singlelevels_1980.nc'

In [2]:
import os
import xarray as xr
import pandas as pd
import json

DATA_DIR = '../../data/ERA5'

# Open the NetCDF file
ds = xr.open_dataset(os.path.join(DATA_DIR, 'era5_singlelevels_1980.nc'))

# Save metadata
metadata = {
    'dimensions': {k: v for k, v in ds.dims.items()},
    'variables': {},
    'global_attributes': dict(ds.attrs)
}

for var in ds.variables:
    metadata['variables'][var] = {
        'dims': list(ds[var].dims),
        'shape': list(ds[var].shape),
        'dtype': str(ds[var].dtype),
        'attributes': dict(ds[var].attrs)
    }

meta_path = os.path.join(DATA_DIR, 'era5_singlelevels_1980_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"Metadata saved to {meta_path}")

# Convert to DataFrame and save as CSV
csv_path = os.path.join(DATA_DIR, 'era5_singlelevels_1980.csv')
df = ds.to_dataframe().reset_index()
df.to_csv(csv_path, index=False)

print(f"CSV saved to {csv_path} ({len(df)} rows)")

ds.close()


Metadata saved to ../../data/ERA5\era5_singlelevels_1980_metadata.json
CSV saved to ../../data/ERA5\era5_singlelevels_1980.csv (30132 rows)


C:\Users\sthar\AppData\Local\Temp\ipykernel_6312\4115340164.py:13: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dimensions': {k: v for k, v in ds.dims.items()},


In [3]:
import os
import pickle
import mimetypes
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

SCOPES = ['https://www.googleapis.com/auth/drive']
TOKEN_PATH = '../../token.pickle'
CLIENT_SECRETS = '../../client_secrets.json'

# Parent Drive folder under which all data folders are uploaded
DRIVE_PARENT_ID = '1zLJgkYkrM1LRgtl7v5sfAQ4aits9zfUq'

# Local data root containing one or more subfolders (ERA5, and future ones)
DATA_ROOT = '../../data'


def get_drive_service():
    creds = None
    if os.path.exists(TOKEN_PATH):
        with open(TOKEN_PATH, 'rb') as f:
            creds = pickle.load(f)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CLIENT_SECRETS, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(TOKEN_PATH, 'wb') as f:
            pickle.dump(creds, f)
    return build('drive', 'v3', credentials=creds)


def get_or_create_folder(service, name, parent_id):
    """Return the Drive folder id for `name` under `parent_id`, creating it if needed."""
    query = (
        f"name = '{name}' and mimeType = 'application/vnd.google-apps.folder' "
        f"and '{parent_id}' in parents and trashed = false"
    )
    response = service.files().list(q=query, spaces='drive', fields='files(id, name)').execute()
    files = response.get('files', [])
    if files:
        return files[0]['id']
    folder = service.files().create(
        body={'name': name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [parent_id]},
        fields='id'
    ).execute()
    return folder['id']


def upload_folder(service, local_dir, parent_id):
    """Recursively upload `local_dir` and its contents under Drive `parent_id`."""
    folder_name = os.path.basename(os.path.normpath(local_dir))
    drive_folder_id = get_or_create_folder(service, folder_name, parent_id)
    print(f"Folder: {folder_name} (Drive id: {drive_folder_id})")

    for entry in sorted(os.listdir(local_dir)):
        local_path = os.path.join(local_dir, entry)
        if os.path.isdir(local_path):
            upload_folder(service, local_path, drive_folder_id)
        else:
            mimetype = mimetypes.guess_type(local_path)[0] or 'application/octet-stream'
            media = MediaFileUpload(local_path, mimetype=mimetype, resumable=True)
            result = service.files().create(
                body={'name': entry, 'parents': [drive_folder_id]},
                media_body=media,
                fields='id, name'
            ).execute()
            print(f"  Uploaded: {result['name']} (id: {result['id']})")


service = get_drive_service()

# Upload every subfolder in the data root (ERA5 now, more in the future)
for entry in sorted(os.listdir(DATA_ROOT)):
    sub = os.path.join(DATA_ROOT, entry)
    if os.path.isdir(sub):
        upload_folder(service, sub, DRIVE_PARENT_ID)

print("Done.")


Folder: ERA5 (Drive id: 1VZmS-_nqz8eyHMx2t2RE3kxs8a9D0aj-)
  Uploaded: era5_singlelevels_1980.csv (id: 10KaS8vSDjZ85hWktzoU3FB51XFCakmGJ)
  Uploaded: era5_singlelevels_1980.nc (id: 1TgzQ0s4f8Bg-LA8oOxusS9Tn5kU49jM6)
  Uploaded: era5_singlelevels_1980_metadata.json (id: 1iKv3800s5WvHhI1vO-HhQBqaJRK0OHOA)
Done.
